<a href="https://colab.research.google.com/github/C-1Blue/ds2002-fa26/blob/main/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
# TODO
df['revenue'] = df['qty'] * df['price']
df.head()
print(len(df))
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

total_revenue, total_units


400


(np.float64(8520.0), np.int64(783))

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [25]:
# TODO
by_category = (
    df.groupby('category', as_index=False)
      .agg(revenue=('revenue','sum'))
)


revenue = by_category['revenue'].sum()
by_category['share_pct'] = (by_category['revenue'] / revenue* 100).round(1)


by_category = by_category.sort_values('revenue', ascending=False)

by_category

,category,revenue,share_pct
1,Food,4293.0,50.4
2,Merch,1771.5,20.8
0,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
# TODO
vendorstats = (
    df.groupby('vendor_id')
      .agg(
          avg_revenue=('revenue', 'mean'),
          order_count=('revenue', 'count')) .sort_values('avg_revenue', ascending=False)
)

vendorstats


,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
# TODO
merchrev = df.loc[df['category'] == 'Merch', 'revenue'].sum()
total_rev = df['revenue'].sum()

sharemerch = (merchrev / total_rev * 100).round(1)
sharemerch


np.float64(20.8)

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [28]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)
df.shape[0], joined.shape[0]
df['revenue'].sum(), joined['revenue'].sum()
unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
unmatched
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown Vendor')
joined


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown Vendor
2,V-18,Drink,3,4.5,13.5,Unknown Vendor
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown Vendor
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown Vendor
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unknown Vendor


**The unmatched vendor, and what I did about it:** _I labeled the unmatched vendor as unknown vendor and the unmatched vendor was V-18._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
# TODO
pivot = pd.pivot_table(
    df,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    fill_value=0
)
pivot['Total'] = pivot.sum(axis=1)
pivot.loc['Total'] = pivot.sum(axis=0)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [29]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_Vendors with lower average order should try to revist their products and raise the price of each order, while the best selling vendors could focus on leaning into selling more items that already sell well. V-10's average order was around 24 which is higher than most vendors being towards 17-21. With that information, there is room to upsell certain items or orders.

The least trustworthy result is probably the vendor by category pivot table beacause V-18 wasn't in the lookup. Because it was labeled as unknown, it weakens the interpretation on vendor performance.  _